# 10 — Final Kadak Bhav Recommendation Engine
Task 11: not another ML model — the final decision layer that combines Tasks 6–10 (price, logistics, perishability, buyer reliability) into one explainable market recommendation.

## Cell 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import os

## Cell 2 — Load all required outputs

In [ ]:
price_data = pd.read_csv(
    "../data/processed/market_performance.csv"
)

logistics_data = pd.read_csv(
    "../data/processed/real_world_market_recommendations.csv"
)

reliability_data = pd.read_csv(
    "../data/processed/market_reliability.csv"
)

print("Price:", price_data.shape)
print("Logistics:", logistics_data.shape)
print("Reliability:", reliability_data.shape)

## Cell 3 — Inspect columns
Use your actual Task 9/10 output column names rather than blindly assuming them.

In [ ]:
print("PRICE COLUMNS")
print(price_data.columns.tolist())

print("\nLOGISTICS COLUMNS")
print(logistics_data.columns.tolist())

print("\nRELIABILITY COLUMNS")
print(reliability_data.columns.tolist())

## Cell 4 — Merge price + logistics

In [ ]:
final_df = logistics_data.merge(
    price_data[
        [
            "market",
            "predicted_avg"
        ]
    ],
    on="market",
    how="left",
    suffixes=("", "_price")
)

final_df.head()

## Cell 5 — Add buyer reliability
Now each market carries price + logistics + buyer reliability.

In [ ]:
final_df = final_df.merge(
    reliability_data[
        [
            "market",
            "avg_reliability",
            "risk_level"
        ]
    ],
    on="market",
    how="left"
)

## Cell 6 — Validate the merged dataset

In [ ]:
required_columns = [
    "market",
    "predicted_avg",
    "transport_cost",
    "perishability_penalty",
    "avg_reliability"
]

missing = [
    column
    for column in required_columns
    if column not in final_df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

print("All required columns available.")

## Cell 7 — Farmer input
For the prototype only — later this comes from the frontend.

In [ ]:
farmer_input = {
    "crop": "tomato",
    "quantity_kg": 1000
}

print(farmer_input)

## Cell 8 — Calculate predicted revenue

In [ ]:
quantity_quintal = (
    farmer_input["quantity_kg"] / 100
)

final_df["predicted_revenue"] = (
    final_df["predicted_avg"]
    * quantity_quintal
)

## Cell 9 — Calculate reliability penalty
Uses the Task 10 reliability score.

In [ ]:
risk_weight = 0.10

final_df["reliability_penalty"] = (
    (
        1
        - final_df["avg_reliability"] / 100
    )
    * risk_weight
    * final_df["predicted_revenue"]
)

## Cell 10 — Calculate final net realization
The core formula.

In [ ]:
final_df["net_realization"] = (
    final_df["predicted_revenue"]

    - final_df["transport_cost"]

    - final_df["perishability_penalty"]

    - final_df["reliability_penalty"]
)

final_df["net_per_quintal"] = (
    final_df["net_realization"]
    / quantity_quintal
)

## Cell 11 — Market ranking

In [ ]:
final_df = (
    final_df
    .sort_values(
        "net_realization",
        ascending=False
    )
    .reset_index(drop=True)
)

final_df["rank"] = (
    final_df.index + 1
)

## Cell 12 — Confidence / recommendation status
Don't claim the recommendation is equally reliable everywhere — tie confidence to available buyer reliability.

In [ ]:
def recommendation_status(row):

    if row["avg_reliability"] >= 80:
        return "High Confidence"

    elif row["avg_reliability"] >= 60:
        return "Moderate Confidence"

    return "Low Confidence"

In [ ]:
final_df["recommendation_status"] = (
    final_df.apply(
        recommendation_status,
        axis=1
    )
)

## Cell 13 — Final recommendation function
The function your backend will eventually call.

In [ ]:
def recommend_market(
    data,
    crop,
    quantity_kg,
    top_n=5
):

    result = data.copy()

    quantity_quintal = (
        quantity_kg / 100
    )

    result["predicted_revenue"] = (
        result["predicted_avg"]
        * quantity_quintal
    )

    result["reliability_penalty"] = (
        (
            1
            - result["avg_reliability"] / 100
        )
        * 0.10
        * result["predicted_revenue"]
    )

    result["net_realization"] = (
        result["predicted_revenue"]
        - result["transport_cost"]
        - result["perishability_penalty"]
        - result["reliability_penalty"]
    )

    result["net_per_quintal"] = (
        result["net_realization"]
        / quantity_quintal
    )

    result = result.sort_values(
        "net_realization",
        ascending=False
    )

    result["rank"] = (
        range(1, len(result) + 1)
    )

    return result.head(top_n)

## Cell 14 — Test the engine

In [ ]:
recommendations = recommend_market(
    final_df,
    crop="tomato",
    quantity_kg=1000,
    top_n=5
)

recommendations

## Cell 15 — Produce farmer-friendly recommendation

In [ ]:
best = recommendations.iloc[0]

print("================================")
print("       KADAK BHAV")
print("================================")

print(
    f"Crop: {farmer_input['crop']}"
)

print(
    f"Quantity: "
    f"{farmer_input['quantity_kg']} kg"
)

print(
    f"\n🏆 Recommended Market: "
    f"{best['market']}"
)

print(
    f"Expected Net Realization: "
    f"₹{best['net_realization']:.2f}"
)

print(
    f"Expected Net / Quintal: "
    f"₹{best['net_per_quintal']:.2f}"
)

print(
    f"Buyer Reliability: "
    f"{best['avg_reliability']:.1f}/100"
)

print(
    f"Risk Level: "
    f"{best['risk_level']}"
)

## Cell 16 — Generate explanation
Important for an explainable AI system — not just "go to Market A", but why.

In [ ]:
def explain_recommendation(row):

    reasons = []

    reasons.append(
        f"Predicted price: "
        f"₹{row['predicted_avg']:.2f}"
    )

    reasons.append(
        f"Transport cost: "
        f"₹{row['transport_cost']:.2f}"
    )

    reasons.append(
        f"Perishability penalty: "
        f"₹{row['perishability_penalty']:.2f}"
    )

    reasons.append(
        f"Buyer reliability: "
        f"{row['avg_reliability']:.1f}/100"
    )

    reasons.append(
        f"Expected net realization: "
        f"₹{row['net_realization']:.2f}"
    )

    return reasons

In [ ]:
reasons = explain_recommendation(
    recommendations.iloc[0]
)

for reason in reasons:
    print("•", reason)

## Cell 17 — Final comparison table

In [ ]:
display_columns = [
    "rank",
    "market",
    "predicted_avg",
    "transport_cost",
    "perishability_penalty",
    "avg_reliability",
    "net_realization",
    "net_per_quintal",
    "risk_level"
]

recommendations[
    display_columns
]

## Cell 18 — Save final recommendation

In [ ]:
output_path = (
    "../data/processed/"
    "kadak_bhav_final_recommendations.csv"
)

recommendations.to_csv(
    output_path,
    index=False
)

print(
    f"Saved: {output_path}"
)

## Cell 19 — Create API-ready JSON
This object becomes the bridge between ML → backend → frontend.

In [ ]:
best_market = recommendations.iloc[0]

api_response = {
    "crop": farmer_input["crop"],
    "quantity_kg": farmer_input["quantity_kg"],
    "recommended_market": best_market["market"],
    "predicted_price": float(
        best_market["predicted_avg"]
    ),
    "net_realization": float(
        best_market["net_realization"]
    ),
    "net_per_quintal": float(
        best_market["net_per_quintal"]
    ),
    "buyer_reliability": float(
        best_market["avg_reliability"]
    ),
    "risk_level": best_market["risk_level"]
}

api_response

## Next step
This `api_response` shape is exactly what the FastAPI backend (`api/main.py`) should return from a `/recommend` endpoint: wrap `recommend_market()` and `explain_recommendation()` in a route that takes crop/quantity/location from the request body instead of the hard-coded `farmer_input` used here, and load the three source CSVs (or, eventually, live data) at startup instead of per-request.